In [ ]:
import pandas as pd
df=pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.head())

In [ ]:
df.shape
print(df.shape)

In [ ]:
df.info()
print(df.info())

In [ ]:
clean_df=pd.read_csv("../data/processed/telco_churn_clean.csv")
print("Raw dataset shape;",df.shape)
print("Cleaned dataset shape;",clean_df.shape)
print("\nRaw TotalCharges type:")
print(df["TotalCharges"].dtype)
print("\nCleaned TotalCharges type:")
print(clean_df["TotalCharges"].dtype)

In [ ]:
print("Missing values in cleaned TotalCharges:")
print(clean_df["TotalCharges"].isna().sum())

print("\nMissing values in all cleaned columns:")
print(clean_df.isna().sum())

print("\nDuplicate customer IDs:")
print(clean_df["customerID"].duplicated().sum())

In [ ]:
missing_charges = clean_df[clean_df["TotalCharges"].isna()]

missing_charges[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

In [ ]:
raw_missing_values = df.loc[
    clean_df["TotalCharges"].isna(),
    "TotalCharges"
]

raw_missing_values.map(repr).value_counts()

### Data Quality Finding: Missing Total Charges

- The original dataset contains 11 whitespace-only values in `TotalCharges`.
- These values become missing (`NaN`) when converted to a numeric data type.
- All 11 affected customers have a tenure of 0 months.
- No duplicate customer IDs were found.
- The dataset retains all 7,043 customer records.

**Decision:** Retain the affected customer records and preserve `TotalCharges` as missing. Do not automatically replace these values with zero, because the source data does not confirm that zero is the correct total charge.

In [ ]:
print("Duplicate rows:", clean_df.duplicated().sum())
print("Unique customer IDs:", clean_df["customerID"].nunique())
print("Rows in cleaned dataset:", len(clean_df))
print("Missing TotalCharges:", clean_df["TotalCharges"].isna().sum())

In [ ]:
clean_df["Churn"].value_counts()
print(clean_df["Churn"].value_counts())

In [ ]:
clean_df["Churn"].value_counts(normalize=True) * 100
print(clean_df["Churn"].value_counts(normalize=True) * 100)

In [ ]:
churn_counts = clean_df["Churn"].value_counts()

churn_percentages = (
    clean_df["Churn"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Customer counts:")
print(churn_counts)

print("\nChurn percentages:")
print(churn_percentages)

In [ ]:
clean_df.groupby("Churn")["tenure"].mean().round(2)

In [ ]:
clean_df.groupby("Churn")["MonthlyCharges"].agg(
    ["mean", "median"]
).round(2)

### EDA Finding: Monthly Charges

Customers who churned had higher average monthly charges (74.44) than customers who stayed (61.27). Their median monthly charges were also higher (79.65 compared with 64.43).

**Business implication:** Investigate whether pricing, service combinations, or contract arrangements are associated with higher churn. These descriptive statistics do not establish causation.

In [ ]:
clean_df.groupby("Churn")["TotalCharges"].agg(
    ["count", "mean", "median"]
).round(2)

In [ ]:
import matplotlib.pyplot as plt
clean_df.boxplot(column="tenure", by="Churn")

plt.title("Customer Tenure by Churn Status")
plt.suptitle("")
plt.xlabel("Churn")
plt.ylabel("Tenure (months)")
plt.show()

In [ ]:
avg_tenure = (
    clean_df.groupby("Churn")["tenure"]
    .mean()
    .reindex(["No", "Yes"])
)

avg_tenure.index = ["Stayed", "Churned"]

avg_tenure.plot(kind="bar", color=["steelblue", "tomato"])

plt.title("Average Tenure: Stayed vs Churned")
plt.xlabel("Churn Status")
plt.ylabel("Average Tenure (Months)")
plt.xticks(rotation=0)
for container in plt.gca().containers:
    plt.bar_label(container, fmt="%.2f", padding=3)
plt.tight_layout()
plt.show()

Overall churn: 26.54% (1,869 of 7,043 customers).

Tenure: Customers who churned had an average tenure of 17.98 months, compared with 37.57 months for customers who stayed.

Monthly charges: Average charges were 74.44 among churned customers and 61.27 among retained customers.

Total charges: Average total charges were lower among churned customers (1,531.80 versus 2,555.34), potentially reflecting shorter customer relationships.

Data quality: 11 missing TotalCharges values were traced to whitespace in the raw CSV. The affected records were retained.